# CMPE 58T — Turkish Dependency Parsing
## 05a — Morphological Feature Injection: UD FEATS (V2)

**V1:** `baseline_boun.pt` evaluated directly with `parser.evaluate()`
**V2:** BERTurk + UD FEATS injection (dim=32, sum pooling, monkey-patched forward)

**BATCH_SIZE=2000** — reduced from 5000 to fit BERTurk + morph modules in 15 GB GPU memory.
`gc.collect() + torch.cuda.empty_cache()` called before training to free any stale allocations.

---
## 0 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
MORPH_DIR    = os.path.join(MODEL_DIR,   "morph")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, MORPH_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

MODEL_V1       = os.path.join(MODEL_DIR,  "baseline_boun.pt")
MODEL_V2       = os.path.join(MORPH_DIR, "morph_ud_feats_boun.pt")
MORPH_STATE_V2 = MODEL_V2 + ".morph_state.pt"

if not os.path.exists(MODEL_V1):
    raise FileNotFoundError("baseline_boun.pt not found. Run 02_baseline.ipynb first.")
print(f"baseline_boun.pt : OK ({os.path.getsize(MODEL_V1):,} B)")
print(f"morph_ud_feats   : {'exists' if os.path.exists(MODEL_V2) else 'new — will train'}")


Mounted at /content/drive
baseline_boun.pt : OK (457,210,255 B)
morph_ud_feats   : new — will train


---
## 1 — Install & Patch

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q

import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))"
NEW = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)"
with open(PARSER_PY, "r") as f: src = f.read()
if OLD in src:
    with open(PARSER_PY, "w") as f: f.write(src.replace(OLD, NEW))
    print("Patch applied: torch.load weights_only=False")
else:
    print("Patch already applied")

from transformers import AdamW, get_linear_schedule_with_warmup
import torch, transformers
print(f"transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch runtime!'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 43.4 MB/s 

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


transformers : 4.35.2
PyTorch      : 2.10.0+cu128
GPU          : NVIDIA A100-SXM4-40GB


---
## 2 — Hyperparameters

In [3]:
import torch

BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

MORPH_DIM  = 32
MAX_FEATS  = 16

HIDDEN_SIZE  = 800
DROPOUT      = 0.33
# Batch size reduced from 5000: BERTurk + morph modules exceed GPU memory at 5000
BATCH_SIZE   = 2000
MAX_EPOCHS   = 30
PATIENCE     = 10
LR           = 1e-3
BERT_LR      = 2e-5
WARMUP       = 0.1
WEIGHT_DECAY = 0.01
EPS          = 1e-8
GRAD_CLIP    = 5.0

SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=GRAD_CLIP
)

print(f"LR={LR}  BERT_LR={BERT_LR}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}")
print(f"MORPH_DIM={MORPH_DIM}  MAX_FEATS={MAX_FEATS}  BATCH_SIZE={BATCH_SIZE}  DEVICE={DEVICE}")


LR=0.001  BERT_LR=2e-05  MAX_EPOCHS=30  PATIENCE=10
MORPH_DIM=32  MAX_FEATS=16  BATCH_SIZE=2000  DEVICE=cuda


---
## 3 — Feature Vocabulary

In [4]:
import json, conllu
from collections import Counter

feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")

if os.path.exists(feat_vocab_path):
    with open(feat_vocab_path, "r", encoding="utf-8") as f:
        FEAT_VOCAB = json.load(f)
    print(f"Loaded feat_vocab.json: {len(FEAT_VOCAB)} entries")
else:
    print("feat_vocab.json not found — rebuilding from all treebanks...")
    val_counts = Counter()
    for path in [BOUN["train"], BOUN["dev"], BOUN["test"],
                 IMST["train"], IMST["dev"], IMST["test"], PUD["test"]]:
        with open(path, encoding="utf-8") as f:
            sents = conllu.parse(f.read())
        for sent in sents:
            for tok in sent:
                if isinstance(tok["id"], int) and tok["feats"]:
                    for k, v in tok["feats"].items():
                        val_counts[f"{k}={v}"] += 1
    FEAT_VOCAB = {kv: idx for idx, (kv, _) in enumerate(val_counts.most_common())}
    FEAT_VOCAB["<PAD>"] = len(FEAT_VOCAB)
    FEAT_VOCAB["<UNK>"] = len(FEAT_VOCAB)
    with open(feat_vocab_path, "w", encoding="utf-8") as f:
        json.dump(FEAT_VOCAB, f, ensure_ascii=False, indent=2)
    print(f"Built and saved: {len(FEAT_VOCAB)} entries")

PAD_IDX    = FEAT_VOCAB["<PAD>"]
UNK_IDX    = FEAT_VOCAB["<UNK>"]
VOCAB_SIZE = len(FEAT_VOCAB)
print(f"  VOCAB_SIZE={VOCAB_SIZE}  PAD={PAD_IDX}  UNK={UNK_IDX}")


Loaded feat_vocab.json: 87 entries
  VOCAB_SIZE=87  PAD=85  UNK=86


---
## 4 — Morphological Modules

In [5]:
import torch
import torch.nn as nn
import types

class MorphEmbedder(nn.Module):
    """
    Embedding table with sum pooling per token.
    Input : [batch, seq_len, max_feats]  LongTensor
    Output: [batch, seq_len, morph_dim]
    """
    def __init__(self, vocab_size, morph_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, morph_dim, padding_idx=pad_idx)
        nn.init.normal_(self.embedding.weight, std=0.01)
        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)

    def forward(self, indices):
        return self.embedding(indices).sum(dim=2)


class MorphProjection(nn.Module):
    """Projects [bert_out || morph_emb] -> n_hidden."""
    def __init__(self, n_hidden, morph_dim):
        super().__init__()
        self.proj = nn.Linear(n_hidden + morph_dim, n_hidden)
        nn.init.eye_(self.proj.weight[:n_hidden, :n_hidden])
        nn.init.zeros_(self.proj.weight[:, n_hidden:])
        nn.init.zeros_(self.proj.bias)

    def forward(self, bert_out, morph_emb):
        return self.proj(torch.cat([bert_out, morph_emb], dim=-1))


FEATS_COL = 5  # CoNLL-U FEATS column index in SuPar CoNLLSentence.values

def batch_to_morph_tensor(batch, feat_vocab, pad_idx, unk_idx, max_feats, device):
    """
    Convert SuPar batch FEATS column to padded LongTensor.
    Position 0 is BOS — left as PAD.
    Returns: [batch_size, seq_len, max_feats]
    """
    batch_size = len(batch.sentences)
    seq_len    = max(len(s.values[FEATS_COL]) + 1 for s in batch.sentences)
    morph = torch.full((batch_size, seq_len, max_feats), pad_idx, dtype=torch.long)
    for b, sent in enumerate(batch.sentences):
        for t, feat_str in enumerate(sent.values[FEATS_COL], start=1):
            if feat_str == "_" or not feat_str:
                continue
            for f_idx, pair in enumerate(feat_str.split("|")[:max_feats]):
                morph[b, t, f_idx] = feat_vocab.get(pair.strip(), unk_idx)
    return morph.to(device)


def patch_model_forward(parser, morph_embedder, morph_projection):
    """
    Monkey-patch parser.model.forward() to accept morph_indices.
    If morph_indices=None -> standard behavior (no morph injection).
    """
    from supar.utils.common import MIN
    _embedder   = morph_embedder
    _projection = morph_projection
    _encode     = parser.model.encode

    def forward_with_morph(self, words, feats=None, morph_indices=None):
        x = _encode(words, feats)
        if morph_indices is not None:
            x = _projection(x, _embedder(morph_indices))
        x = self.encoder_dropout(x)
        mask  = words.ne(self.args.pad_index) if len(words.shape) < 3 else words.ne(self.args.pad_index).any(-1)
        arc_d = self.arc_mlp_d(x);  arc_h = self.arc_mlp_h(x)
        sib_s = self.sib_mlp_s(x);  sib_d = self.sib_mlp_d(x);  sib_h = self.sib_mlp_h(x)
        rel_d = self.rel_mlp_d(x);  rel_h = self.rel_mlp_h(x)
        s_arc = self.arc_attn(arc_d, arc_h).masked_fill_(~mask.unsqueeze(1), MIN)
        s_sib = self.sib_attn(sib_s, sib_d, sib_h).permute(0, 3, 1, 2)
        s_rel = self.rel_attn(rel_d, rel_h).permute(0, 2, 3, 1)
        return s_arc, s_sib, s_rel

    parser.model.forward = types.MethodType(forward_with_morph, parser.model)
    print("Model forward patched: morph_indices argument supported.")


print("MorphEmbedder, MorphProjection, batch_to_morph_tensor, patch_model_forward defined.")


MorphEmbedder, MorphProjection, batch_to_morph_tensor, patch_model_forward defined.


---
## 5 — Load Parser & Initialize Modules

In [6]:
from supar import CRF2oDependencyParser

if os.path.exists(MODEL_V2):
    print(f"V2 checkpoint found — loading: {MODEL_V2}")
    parser = CRF2oDependencyParser.load(MODEL_V2)
else:
    print(f"Loading baseline_boun.pt as starting point...")
    parser = CRF2oDependencyParser.load(MODEL_V1)

parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
n_hidden = parser.model.encoder.n_out

morph_embedder   = MorphEmbedder(VOCAB_SIZE, MORPH_DIM, PAD_IDX).to(DEVICE)
morph_projection = MorphProjection(n_hidden, MORPH_DIM).to(DEVICE)

# Load saved morph weights if V2 checkpoint exists
if os.path.exists(MORPH_STATE_V2):
    state = torch.load(MORPH_STATE_V2, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Morph module weights loaded from checkpoint.")

patch_model_forward(parser, morph_embedder, morph_projection)

morph_params = (sum(p.numel() for p in morph_embedder.parameters()) +
                sum(p.numel() for p in morph_projection.parameters()))
print(f"  n_hidden={n_hidden}  morph_params={morph_params:,}  device={DEVICE}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


Loading baseline_boun.pt as starting point...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Model forward patched: morph_indices argument supported.
  n_hidden=768  morph_params=617,952  device=cuda
  GPU memory: 0.46 GB allocated


---
## 6 — Data Loaders & Optimizer

In [7]:
from supar.utils import Dataset
from transformers import AdamW, get_linear_schedule_with_warmup

def make_loader(parser, path, batch_size, shuffle=False):
    dataset = Dataset(parser.transform, path)
    dataset.build(batch_size, 32)
    return dataset.loader

print(f"Building data loaders (batch_size={BATCH_SIZE})...")
train_loader = make_loader(parser, BOUN["train"], BATCH_SIZE, shuffle=True)
dev_loader   = make_loader(parser, BOUN["dev"],   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(parser, BOUN["test"],  BATCH_SIZE, shuffle=False)
pud_loader   = make_loader(parser, PUD["test"],   BATCH_SIZE, shuffle=False)
print(f"  train={len(train_loader)}  dev={len(dev_loader)}  test={len(test_loader)}  pud={len(pud_loader)} batches")

total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP)

bert_params = list(parser.model.encoder.parameters())
task_params = (
    [p for p in parser.model.parameters() if not any(p is bp for bp in bert_params)] +
    list(morph_embedder.parameters()) +
    list(morph_projection.parameters())
)
all_params = bert_params + task_params

optimizer = AdamW(
    [{"params": bert_params, "lr": BERT_LR},
     {"params": task_params, "lr": LR}],
    weight_decay=WEIGHT_DECAY, eps=EPS
)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"  total_steps={total_steps}  warmup={warmup_steps}")


Building data loaders (batch_size=2000)...


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


  train=70  dev=32  test=32  pud=32 batches


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  total_steps=2100  warmup=210


---
## 7 — Train V2

Skips training if checkpoint already exists on Drive.
Calls `gc.collect() + torch.cuda.empty_cache()` before training starts.

In [12]:
import torch.nn as nn
import gc
from supar.utils.metric import AttachmentMetric

def run_epoch(parser, loader, morph_embedder, morph_projection,
              optimizer, scheduler, device, use_morph, training=True):
    if training:
        parser.model.train(); morph_embedder.train(); morph_projection.train()
    else:
        parser.model.eval(); morph_embedder.eval(); morph_projection.eval()

    total_loss = 0.0
    metric     = AttachmentMetric()
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch in loader:
            # Unpack: last 3 are always arcs, sibs, rels.
            # Second element may be 'texts' (strings, not a tensor) — skip it.
            # First element is always words.
            items = list(batch)
            words = items[0]
            # Skip non-tensor items (texts is a list of strings)
            tensor_items = [x for x in items if hasattr(x, 'to')]
            # tensor_items: [words, ...feats..., arcs, sibs, rels]
            arcs  = tensor_items[-3]
            sibs  = tensor_items[-2]
            rels  = tensor_items[-1]
            feats = tensor_items[1:-3]

            word_mask = words.ne(parser.args.pad_index)
            mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(device)
            feats = [f.to(device) for f in feats]
            arcs  = arcs.to(device);  sibs = sibs.to(device)
            rels  = rels.to(device);  mask = mask.to(device)

            morph_indices = None
            if use_morph:
                morph_indices = batch_to_morph_tensor(
                    batch, FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, device)
                morph_indices = morph_indices[:, :words.shape[1], :]

            s_arc, s_sib, s_rel = parser.model(words, feats, morph_indices)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)

            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
                optimizer.step(); scheduler.step()

            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric


gc.collect()
torch.cuda.empty_cache()

if os.path.exists(MODEL_V2) and os.path.exists(MORPH_STATE_V2):
    print("V2 checkpoint exists — skipping training.")
    print("Delete morph_ud_feats_boun.pt and .morph_state.pt to retrain.")
else:
    print(f"Training V2: BERTurk + UD FEATS | batch={BATCH_SIZE} epochs={MAX_EPOCHS} patience={PATIENCE}")
    print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
    print("=" * 65)

    best_las = 0.0; patience_ct = 0; history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, _ = run_epoch(parser, train_loader, morph_embedder, morph_projection,
                                   optimizer, scheduler, DEVICE, use_morph=True, training=True)
        dev_loss, dev_m = run_epoch(parser, dev_loader, morph_embedder, morph_projection,
                                     optimizer, scheduler, DEVICE, use_morph=True, training=False)
        uas = dev_m.uas * 100; las = dev_m.las * 100
        print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  train {train_loss:.4f} | dev {dev_loss:.4f} | UAS {uas:.2f} LAS {las:.2f}", end="")

        if las > best_las:
            best_las = las; patience_ct = 0
            torch.save({"morph_embedder": morph_embedder.state_dict(),
                        "morph_projection": morph_projection.state_dict()},
                       MORPH_STATE_V2)
            parser.save(MODEL_V2)
            print(" (saved)", end="")
        else:
            patience_ct += 1
            print(f" (patience {patience_ct}/{PATIENCE})", end="")
        print()
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "dev_loss": dev_loss, "dev_uas": uas, "dev_las": las})
        if patience_ct >= PATIENCE:
            print(f"Early stopping at epoch {epoch}."); break

    print(f"\nBest dev LAS: {best_las:.2f}")
    with open(os.path.join(RESULTS_DIR, "morph_v2_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print("History saved.")

Training V2: BERTurk + UD FEATS | batch=2000 epochs=30 patience=10
GPU memory: 0.46 GB allocated
Epoch  1/30  train 48.0683 | dev 29.1789 | UAS 23.26 LAS 13.13 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  2/30  train 4.3292 | dev 2.4351 | UAS 73.07 LAS 58.67 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  3/30  train 1.0413 | dev 2.0406 | UAS 76.11 LAS 66.12 (saved)


...

Epoch 12/30  train 0.3763 | dev 2.0692 | UAS 80.32 LAS 73.07 (patience 1/10)
Epoch 13/30  train 0.3521 | dev 2.1777 | UAS 80.69 LAS 73.40 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch 14/30  train 0.3325 | dev 2.0199 | UAS 80.41 LAS 73.12 (patience 1/10)
Epoch 15/30  train 0.3110 | dev 2.2378 | UAS 80.48 LAS 73.07 (patience 2/10)
Epoch 16/30  train 0.3149 | dev 2.2873 | UAS 80.40 LAS 73.04 (patience 3/10)
Epoch 17/30  train 0.8639 | dev 2.3977 | UAS 80.27 LAS 72.93 (patience 4/10)
Epoch 18/30  train 0.2431 | dev 2.5605 | UAS 80.52 LAS 73.48 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch 19/30  train 1.0393 | dev 2.3811 | UAS 80.41 LAS 72.71 (patience 1/10)
Epoch 20/30  train 0.1907 | dev 2.7545 | UAS 80.45 LAS 73.04 (patience 2/10)
Epoch 21/30  train 0.0899 | dev 2.9911 | UAS 80.32 LAS 72.73 (patience 3/10)
Epoch 22/30  train -0.0135 | dev 3.0137 | UAS 80.35 LAS 72.90 (patience 4/10)
Epoch 23/30  train -0.0498 | dev 3.5236 | UAS 80.65 LAS 73.46 (patience 5/10)
Epoch 24/30  train -0.6574 | dev 4.1549 | UAS 79.96 LAS 72.78 (patience 6/10)
Epoch 25/30  train -0.8692 | dev 4.1825 | UAS 80.47 LAS 73.28 (patience 7/10)
Epoch 26/30  train -2.7991 | dev 4.8503 | UAS 80.60 LAS 73.35 (patience 8/10)
Epoch 27/30  train -4.5641 | dev 4.9207 | UAS 80.53 LAS 73.27 (patience 9/10)
Epoch 28/30  train -11.9657 | dev 5.4395 | UAS 80.52 LAS 73.23 (patience 10/10)
Early stopping at epoch 28.

Best dev LAS: 73.48
History saved.


---
## 8 — Evaluate V1 and V2

V1 uses `parser.evaluate()` directly (no morph, no patch needed).
V2 uses `AttachmentMetric` from the custom training loop (parser.predict() cannot accept morph_indices).
Both save `.conllu` and `_sent_las.npy` for downstream analysis.

In [21]:
import json, numpy as np
import conllu as conllu_lib
from supar.utils.metric import AttachmentMetric
import torch.nn as nn

# Redefine run_epoch with the ORIGINAL unpacking that worked
def run_epoch(parser, loader, morph_embedder, morph_projection,
              optimizer, scheduler, device, use_morph, training=True):
    if training:
        parser.model.train(); morph_embedder.train(); morph_projection.train()
    else:
        parser.model.eval(); morph_embedder.eval(); morph_projection.eval()

    total_loss = 0.0
    metric     = AttachmentMetric()
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch in loader:
            words, texts, *feats, arcs, sibs, rels = batch
            word_mask = words.ne(parser.args.pad_index)
            mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(device)
            feats = [f.to(device) for f in feats]
            arcs  = arcs.to(device);  sibs = sibs.to(device)
            rels  = rels.to(device);  mask = mask.to(device)

            morph_indices = None
            if use_morph:
                morph_indices = batch_to_morph_tensor(
                    batch, FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, device)
                morph_indices = morph_indices[:, :words.shape[1], :]

            s_arc, s_sib, s_rel = parser.model(words, feats, morph_indices)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)

            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
                optimizer.step(); scheduler.step()

            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric


# Rebuild loaders at BATCH_SIZE=5000 (original batch size that includes texts field)
from supar.utils import Dataset
def make_loader(parser, path, batch_size, shuffle=False):
    dataset = Dataset(parser.transform, path)
    dataset.build(batch_size, 32)
    return dataset.loader

test_loader = make_loader(parser, BOUN["test"], 5000, shuffle=False)
pud_loader  = make_loader(parser, PUD["test"],  5000, shuffle=False)

# Verify
batch = next(iter(test_loader))
items = list(batch)
print(f"Batch length: {len(items)}")
for i, x in enumerate(items):
    if hasattr(x, 'shape'):
        print(f"  [{i}] tensor shape={x.shape}")
    else:
        print(f"  [{i}] type={type(x).__name__}")

Batch length: 2
  [0] tensor shape=torch.Size([79, 8, 5])
  [1] type=list


In [25]:
import json, numpy as np
import conllu as conllu_lib
from collections import defaultdict
import torch
from supar.utils import Dataset

# Load best morph weights
if os.path.exists(MORPH_STATE_V2):
    state = torch.load(MORPH_STATE_V2, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Best morph weights loaded.")

def get_rel_vocab(parser):
    for field in parser.transform.flattened_fields:
        if hasattr(field, 'vocab') and hasattr(field, 'name') \
                and field.name in ('rels', 'rel', 'REL', 'deprel'):
            return field.vocab
    for field in parser.transform.flattened_fields:
        if hasattr(field, 'vocab') and len(field.vocab) > 10:
            return field.vocab
    raise RuntimeError("Cannot find relation vocab")

def run_morph_inference(parser, conllu_path, morph_embedder, morph_projection,
                        feat_vocab, pad_idx, unk_idx, max_feats, device):
    parser.model.eval()
    morph_embedder.eval()
    morph_projection.eval()

    rel_vocab = get_rel_vocab(parser)
    # Build index→string mapping regardless of vocab type
    if hasattr(rel_vocab, 'itos'):
        idx2rel = rel_vocab.itos        # list: idx2rel[i] = string
    elif isinstance(rel_vocab, dict):
        idx2rel = {v: k for k, v in rel_vocab.items()}  # reverse stoi dict
    else:
        idx2rel = rel_vocab

    dataset = Dataset(parser.transform, conllu_path)
    dataset.build(BATCH_SIZE, 32)

    with open(conllu_path, encoding="utf-8") as f:
        gold_sents = conllu_lib.parse(f.read())
    with open(conllu_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())

    sent_idx = 0
    with torch.no_grad():
        for batch in dataset.loader:
            words = batch[0].to(device)

            morph_indices = batch_to_morph_tensor(
                batch, feat_vocab, pad_idx, unk_idx, max_feats, device)
            morph_indices = morph_indices[:, :words.shape[1], :]

            word_mask = words.ne(parser.args.pad_index)
            mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0

            s_arc, s_sib, s_rel = parser.model(words, [], morph_indices)
            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)

            arc_preds = arc_preds.cpu()
            rel_preds = rel_preds.cpu()
            mask_cpu  = mask.cpu()

            for i in range(arc_preds.shape[0]):
                if sent_idx >= len(pred_sents):
                    break
                sent      = pred_sents[sent_idx]
                real_toks = [t for t in sent if isinstance(t["id"], int)]
                valid_len = int(mask_cpu[i].sum().item())
                for j in range(min(valid_len, len(real_toks))):
                    real_toks[j]["head"]   = int(arc_preds[i, j+1].item())
                    real_toks[j]["deprel"] = idx2rel[int(rel_preds[i, j+1].item())]
                sent_idx += 1

    # Score against gold
    total = correct_arc = correct_both = 0
    sent_las = []
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        n = len(gt)
        if n == 0:
            continue
        arc_ok  = sum(1 for g, p in zip(gt, pt) if g["head"] == p["head"])
        both_ok = sum(1 for g, p in zip(gt, pt)
                      if g["head"] == p["head"] and g["deprel"] == p["deprel"])
        total        += n
        correct_arc  += arc_ok
        correct_both += both_ok
        sent_las.append(both_ok == n)

    uas = round(100 * correct_arc  / total, 2) if total else 0.0
    las = round(100 * correct_both / total, 2) if total else 0.0
    return pred_sents, uas, las, np.array(sent_las, dtype=bool)


print("Running morph-injected inference on BOUN test...")
pred_sents, uas, las, sent_las_arr = run_morph_inference(
    parser, BOUN["test"], morph_embedder, morph_projection,
    FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, DEVICE)

print(f"  V2 BOUN → UAS {uas:.2f}  LAS {las:.2f}")

pred_conllu = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN.conllu")
with open(pred_conllu, "w", encoding="utf-8") as f:
    for sent in pred_sents:
        f.write(sent.serialize())
print(f"  Saved: {os.path.basename(pred_conllu)}")

pred_npy = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN_sent_las.npy")
np.save(pred_npy, sent_las_arr)
print(f"  Saved: {os.path.basename(pred_npy)}")

results = [
    {"system": "v1_baseline",       "test": "BOUN",
     "UAS": 77.26, "LAS": 61.96, "pred_conllu": "-", "pred_npy": "-"},
    {"system": "morph_v2_ud_feats", "test": "BOUN",
     "UAS": uas, "LAS": las,
     "pred_conllu": os.path.basename(pred_conllu),
     "pred_npy":    os.path.basename(pred_npy)},
]

results_path = os.path.join(RESULTS_DIR, "morph_v2_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")
print(f"Delta V2 - V1 (BOUN): ΔUAS {uas-77.26:+.2f}  ΔLAS {las-61.96:+.2f}")

Best morph weights loaded.
Running morph-injected inference on BOUN test...
  V2 BOUN → UAS 45.03  LAS 35.70
  Saved: morph_v2_ud_feats_BOUN.conllu
  Saved: morph_v2_ud_feats_BOUN_sent_las.npy

Results saved: /content/drive/MyDrive/58T - Application Project/results/morph_v2_results.json
Delta V2 - V1 (BOUN): ΔUAS -32.23  ΔLAS -26.26


In [26]:
# Sanity: run WITHOUT morph injection — should match V1 (77.26 UAS)
pred_sents_noморph, uas_no, las_no, _ = run_morph_inference(
    parser, BOUN["test"], morph_embedder, morph_projection,
    FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, DEVICE)
# Temporarily test with use_morph=False by patching

In [27]:
print(f"MODEL_V2 exists: {os.path.exists(MODEL_V2)}  size: {os.path.getsize(MODEL_V2) if os.path.exists(MODEL_V2) else 'N/A'}")
print(f"MORPH_STATE_V2 exists: {os.path.exists(MORPH_STATE_V2)}  size: {os.path.getsize(MORPH_STATE_V2) if os.path.exists(MORPH_STATE_V2) else 'N/A'}")

# Test inference WITHOUT morph — should give ~77 UAS if model is correct
import copy

def run_baseline_inference(parser, conllu_path, device):
    """Run standard inference with NO morph injection."""
    parser.model.eval()
    rel_vocab = get_rel_vocab(parser)
    if hasattr(rel_vocab, 'itos'):
        idx2rel = rel_vocab.itos
    elif isinstance(rel_vocab, dict):
        idx2rel = {v: k for k, v in rel_vocab.items()}
    else:
        idx2rel = rel_vocab

    from supar.utils import Dataset
    dataset = Dataset(parser.transform, conllu_path)
    dataset.build(BATCH_SIZE, 32)

    with open(conllu_path, encoding="utf-8") as f:
        gold_sents = conllu_lib.parse(f.read())
    with open(conllu_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())

    sent_idx = 0
    with torch.no_grad():
        for batch in dataset.loader:
            words    = batch[0].to(device)
            word_mask = words.ne(parser.args.pad_index)
            mask = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            # NO morph — pass None
            s_arc, s_sib, s_rel = parser.model(words, [], None)
            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            arc_preds = arc_preds.cpu()
            rel_preds = rel_preds.cpu()
            mask_cpu  = mask.cpu()
            for i in range(arc_preds.shape[0]):
                if sent_idx >= len(pred_sents): break
                sent      = pred_sents[sent_idx]
                real_toks = [t for t in sent if isinstance(t["id"], int)]
                valid_len = int(mask_cpu[i].sum().item())
                for j in range(min(valid_len, len(real_toks))):
                    real_toks[j]["head"]   = int(arc_preds[i, j+1].item())
                    real_toks[j]["deprel"] = idx2rel[int(rel_preds[i, j+1].item())]
                sent_idx += 1

    total = correct_arc = correct_both = 0
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        n = len(gt)
        if n == 0: continue
        correct_arc  += sum(1 for g, p in zip(gt, pt) if g["head"] == p["head"])
        correct_both += sum(1 for g, p in zip(gt, pt)
                            if g["head"] == p["head"] and g["deprel"] == p["deprel"])
        total += n
    uas = round(100 * correct_arc  / total, 2) if total else 0.0
    las = round(100 * correct_both / total, 2) if total else 0.0
    return uas, las

uas_no, las_no = run_baseline_inference(parser, BOUN["test"], DEVICE)
print(f"No-morph inference → UAS {uas_no:.2f}  LAS {las_no:.2f}")
print("(Should be ~77.26 / 61.96 if V2 model loaded correctly)")
print("(If ~45, the loaded parser has wrong weights)")

MODEL_V2 exists: True  size: 457211605
MORPH_STATE_V2 exists: True  size: 2474452
No-morph inference → UAS 44.68  LAS 36.27
(Should be ~77.26 / 61.96 if V2 model loaded correctly)
(If ~45, the loaded parser has wrong weights)


In [28]:
# Check: what does parser.evaluate() give on BOUN test?
# This uses SuPar's own forward — should reflect actual loaded BERTurk weights
parser.args.update(SUPAR_EXTRA)
_, m = parser.evaluate(BOUN["test"], verbose=False)
print(f"parser.evaluate() → UAS {m.uas*100:.2f}  LAS {m.las*100:.2f}")

# Also check baseline_boun.pt
from supar import CRF2oDependencyParser
p_base = CRF2oDependencyParser.load(MODEL_V1)
p_base.args.update(SUPAR_EXTRA)
_, m2 = p_base.evaluate(BOUN["test"], verbose=False)
print(f"baseline_boun.pt  → UAS {m2.uas*100:.2f}  LAS {m2.las*100:.2f}")
del p_base

parser.evaluate() → UAS 23.55  LAS 12.21
baseline_boun.pt  → UAS 77.26  LAS 61.96


In [29]:
from supar import CRF2oDependencyParser
import gc

gc.collect(); torch.cuda.empty_cache()

# Reload from baseline — BERTurk backbone is shared
print("Reloading parser from baseline_boun.pt...")
parser = CRF2oDependencyParser.load(MODEL_V1)
parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
n_hidden = parser.model.encoder.n_out

# Reinitialize morph modules
morph_embedder   = MorphEmbedder(VOCAB_SIZE, MORPH_DIM, PAD_IDX).to(DEVICE)
morph_projection = MorphProjection(n_hidden, MORPH_DIM).to(DEVICE)

# Load trained morph weights
state = torch.load(MORPH_STATE_V2, map_location=DEVICE, weights_only=False)
morph_embedder.load_state_dict(state["morph_embedder"])
morph_projection.load_state_dict(state["morph_projection"])
print("Morph module weights loaded.")

# Re-patch forward
patch_model_forward(parser, morph_embedder, morph_projection)

# Verify baseline (no morph) still gives 77.26
parser.args.update(SUPAR_EXTRA)
_, m = parser.evaluate(BOUN["test"], verbose=False)
print(f"Baseline check (no morph): UAS {m.uas*100:.2f}  LAS {m.las*100:.2f}")
print("(Should be 77.26 / 61.96)")

Reloading parser from baseline_boun.pt...
Morph module weights loaded.
Model forward patched: morph_indices argument supported.
Baseline check (no morph): UAS 77.26  LAS 61.96
(Should be 77.26 / 61.96)


In [31]:
print("Running morph-injected inference on BOUN test...")
pred_sents, uas, las, sent_las_arr = run_morph_inference(
    parser, BOUN["test"], morph_embedder, morph_projection,
    FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, DEVICE)

print(f"  V2 BOUN → UAS {uas:.2f}  LAS {las:.2f}")

pred_conllu = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN.conllu")
with open(pred_conllu, "w", encoding="utf-8") as f:
    for sent in pred_sents:
        f.write(sent.serialize())
print(f"  Saved: {os.path.basename(pred_conllu)}")

pred_npy = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN_sent_las.npy")
np.save(pred_npy, sent_las_arr)
print(f"  Saved: {os.path.basename(pred_npy)}")

results = [
    {"system": "v1_baseline",       "test": "BOUN",
     "UAS": 77.26, "LAS": 61.96, "pred_conllu": "-", "pred_npy": "-"},
    {"system": "morph_v2_ud_feats", "test": "BOUN",
     "UAS": uas, "LAS": las,
     "pred_conllu": os.path.basename(pred_conllu),
     "pred_npy":    os.path.basename(pred_npy)},
]

results_path = os.path.join(RESULTS_DIR, "morph_v2_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")
print(f"Delta V2 - V1 (BOUN): ΔUAS {uas-77.26:+.2f}  ΔLAS {las-61.96:+.2f}")

Running morph-injected inference on BOUN test...
  V2 BOUN → UAS 43.34  LAS 38.97
  Saved: morph_v2_ud_feats_BOUN.conllu
  Saved: morph_v2_ud_feats_BOUN_sent_las.npy

Results saved: /content/drive/MyDrive/58T - Application Project/results/morph_v2_results.json
Delta V2 - V1 (BOUN): ΔUAS -33.92  ΔLAS -22.99


---
## 9 — Results Table

In [ ]:
print(f"{'System':<25} {'Test':<6} {'UAS':>7} {'LAS':>7}")
print("-" * 50)
for r in results:
    print(f"  {r['system']:<23} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}")

v1_boun = next((r for r in results if r["system"]=="v1_baseline" and r["test"]=="BOUN"), None)
v2_boun = next((r for r in results if "v2" in r["system"] and r["test"]=="BOUN"), None)
if v1_boun and v2_boun:
    d_uas = v2_boun["UAS"] - v1_boun["UAS"]
    d_las = v2_boun["LAS"] - v1_boun["LAS"]
    print(f"\nDelta V2 - V1 (BOUN): ΔUAS {d_uas:+.2f}  ΔLAS {d_las:+.2f}")


---
## 10 — Per-Relation LAS Breakdown (V2 on BOUN)

In [ ]:
import json

pred_path = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN.conllu")
if not os.path.exists(pred_path):
    print("Prediction file not found — run section 8 (Evaluate) first.")
else:
    gold_boun = load_gold(BOUN["test"]) if "load_gold" in dir() else None
    if gold_boun is None:
        import conllu as conllu_lib
        with open(BOUN["test"], encoding="utf-8") as f:
            gold_boun = conllu_lib.parse(f.read())
    with open(pred_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())
    breakdown = per_relation_las(gold_boun, pred_sents)
    br_path = os.path.join(RESULTS_DIR, "morph_v2_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"Per-relation breakdown saved: {br_path}")
    print()
    print(f"{'Relation':<15} {'Count':>7} {'LAS':>7}")
    print("-" * 32)
    for rel, vals in list(breakdown.items())[:20]:
        print(f"  {rel:<13} {vals['total']:>7} {vals['LAS']:>7.2f}")


---
## 11 — Verify Saved Files

In [ ]:
expected = [
    (MORPH_DIR,   "morph_ud_feats_boun.pt"),
    (MORPH_DIR,   "morph_ud_feats_boun.pt.morph_state.pt"),
    (PRED_DIR,    "morph_v2_ud_feats_BOUN.conllu"),
    (PRED_DIR,    "morph_v2_ud_feats_BOUN_sent_las.npy"),
    (PRED_DIR,    "morph_v2_ud_feats_PUD.conllu"),
    (PRED_DIR,    "morph_v2_ud_feats_PUD_sent_las.npy"),
    (RESULTS_DIR, "morph_v2_results.json"),
    (RESULTS_DIR, "morph_v2_history.json"),
    (RESULTS_DIR, "morph_v2_BOUN_per_rel.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 85)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")
print()
if all_ok:
    print("All files present. 05a_morph_ud_feats complete.")
    print("Next: 05b_morph_zemberek.ipynb")
else:
    print("Some files missing — rerun the relevant sections above.")
